# VIX term-structure backtest

Monthly SPY allocation driven by the ratio of a longer-dated Cboe volatility index to a shorter one. The main pair is `VIX3M / VIX` (`3M/1M`): above 1 is contango, below 1 is backwardation.

This notebook only reads saved results. Regenerate them from the repository root first:

```bash
conda activate vol-trading
python -m vol_trading ingest
python -m vol_trading backtest
```

Rules are fixed in `vol_trading/backtest.py`: the signal is read at each month-end close and traded at the next session's close. Cash earns the 3-month T-bill rate, and turnover costs 5 basis points unless `--cost-bps` says otherwise.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "results"
if not (RESULTS / "run_info.json").exists():
    raise FileNotFoundError("Run `python -m vol_trading backtest` from the repository root first.")

info = json.loads((RESULTS / "run_info.json").read_text())
MAIN = f"binary {info['main_pair']}"
HEADLINE = [MAIN, f"zscore {info['main_pair']}", "SPY buy-and-hold", "60/40 SPY/IEF", "VIX < 20 filter"]

def read(name, **kwargs):
    return pd.read_csv(RESULTS / name, **kwargs)

returns = read("headline_returns.csv", index_col=0, parse_dates=True)
spy_weights = read("headline_spy_weights.csv", index_col=0, parse_dates=True)
metrics = read("headline_metrics.csv", index_col=0)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
info

{'main_pair': '3M/1M',
 'cost_bps': 5.0,
 'headline_first_signal': '2010-09-30',
 'pair_grid_first_signal': '2012-01-31',
 'data_end': '2026-10-07'}

## P&L and drawdowns

Growth of $1, log scale. Shaded spans are periods when the main binary rule held cash because the month-end `3M/1M` ratio was at or below 1.

In [ ]:
def cash_spans(weight: pd.Series) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    out = weight < 1.0
    starts = weight.index[out & ~out.shift(1, fill_value=False)]
    ends = weight.index[out & ~out.shift(-1, fill_value=False)]
    return list(zip(starts, ends))

equity = (1.0 + returns[HEADLINE]).cumprod()
drawdown = equity / equity.cummax() - 1.0

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.04,
                    subplot_titles=["Growth of $1 (log scale)", "Drawdown"])
for name in HEADLINE:
    width = 2.5 if name == MAIN else 1.4
    fig.add_trace(go.Scatter(x=equity.index, y=equity[name], name=name, legendgroup=name,
                             line=dict(width=width)), row=1, col=1)
    fig.add_trace(go.Scatter(x=drawdown.index, y=drawdown[name], name=name, legendgroup=name,
                             showlegend=False, line=dict(width=width)), row=2, col=1)
for start, end in cash_spans(spy_weights[MAIN]):
    fig.add_vrect(x0=start, x1=end, fillcolor="gray", opacity=0.15, line_width=0)
fig.update_yaxes(type="log", row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=2, col=1)
fig.update_layout(height=750, title=f"Backtest P&L, first signal {info['headline_first_signal']}, "
                                    f"{info['cost_bps']:g} bps costs", legend=dict(orientation="h", y=1.08))
fig.show()

In [ ]:
summary_columns = ["cagr", "ann_vol", "sharpe", "sortino", "max_drawdown", "calmar", "turnover_per_year", "total_cost"]
display(metrics[summary_columns])

costs = read("cost_sensitivity.csv").pivot_table(index="strategy", columns="cost_bps", values="sharpe")
costs.columns = [f"Sharpe at {c:g} bps" for c in costs.columns]
display(costs.loc[metrics.index])

stress = read("stress.csv", index_col=0)
display((stress.drop(columns=["start", "end"]) * 100).round(1).rename_axis("return in window, %"))

,cagr,ann_vol,sharpe,sortino,max_drawdown,calmar,turnover_per_year,total_cost
strategy,,,,,,,,
binary 3M/1M,0.125,0.136,0.816,1.147,-0.277,0.449,1.686,0.013
zscore 3M/1M,0.070,0.099,0.577,0.791,-0.252,0.280,4.390,0.035
bands 3M/1M,0.114,0.123,0.813,1.128,-0.259,0.441,2.232,0.018
SPY buy-and-hold,0.147,0.169,0.803,1.131,-0.337,0.436,0.062,0.001
60/40 SPY/IEF,0.097,0.097,0.834,1.181,-0.212,0.456,0.282,0.002
VIX < 20 filter,0.073,0.109,0.556,0.765,-0.217,0.336,1.936,0.015
Vol target 15%,0.127,0.137,0.831,1.154,-0.241,0.527,0.986,0.008


,Sharpe at 0 bps,Sharpe at 5 bps,Sharpe at 10 bps
strategy,,,
binary 3M/1M,0.823,0.816,0.810
zscore 3M/1M,0.599,0.577,0.555
bands 3M/1M,0.822,0.813,0.804
SPY buy-and-hold,0.804,0.803,0.803
60/40 SPY/IEF,0.836,0.834,0.833
VIX < 20 filter,0.564,0.556,0.547
Vol target 15%,0.834,0.831,0.827


,binary 3M/1M,zscore 3M/1M,bands 3M/1M,SPY buy-and-hold,60/40 SPY/IEF,VIX < 20 filter,Vol target 15%
"return in window, %",,,,,,,
Aug 2011 debt ceiling,-13.200,-4.900,-9.000,-18.400,-7.300,-4.900,-12.900
Aug 2015 China devaluation,-11.100,-11.100,-11.100,-11.100,-6.400,-11.100,-11.100
Feb 2018 Volmageddon,-10.100,-0.900,-10.100,-10.100,-6.600,-10.100,-10.100
Q4 2018 selloff,-20.900,-13.500,-12.700,-19.300,-10.500,-20.900,-16.900
Mar 2020 COVID crash,0.100,0.100,-4.400,-33.700,-18.700,-8.700,-24.100
2022 bear market,-27.700,-17.000,-24.800,-24.500,-20.700,-4.200,-18.900
Aug 2024 yen carry unwind,-8.400,-3.900,-6.100,-8.400,-3.900,-8.400,-8.400
Apr 2025 tariff shock,-8.200,-4.400,-11.700,-18.800,-10.600,-8.200,-15.800


## Every tenor pair

Each cell is a `long / short` pair; row `6M` and column `1M` is `VIX6M / VIX`. All pairs share one window starting at the first month end where every pair and its z-score exist, so these Sharpe ratios compare with each other but not with the headline table. Read the grid for consistency across pairs, not to pick the best cell.

In [ ]:
pair_metrics = read("pair_metrics.csv")
tenor_order = ["9D", "1M", "3M", "6M", "12M"]
rules = ["binary", "bands", "zscore"]
grid = pair_metrics[pair_metrics["rule"].isin(rules)].copy()
grid[["long", "short"]] = grid["pair"].str.split("/", expand=True)
benchmarks = pair_metrics[pair_metrics["rule"] == "benchmark"].set_index("strategy")["sharpe"]

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{rule} rule" for rule in rules], horizontal_spacing=0.08)
for i, rule in enumerate(rules, start=1):
    table = (grid[grid["rule"] == rule].pivot(index="long", columns="short", values="sharpe")
             .reindex(index=tenor_order[1:], columns=tenor_order[:-1]))
    fig.add_trace(go.Heatmap(z=table.values, x=table.columns, y=table.index, coloraxis="coloraxis",
                             text=np.round(table.values, 2), texttemplate="%{text}"), row=1, col=i)
    fig.update_xaxes(title_text="short tenor", row=1, col=i)
fig.update_yaxes(title_text="long tenor", row=1, col=1)
fig.update_layout(height=420, coloraxis=dict(colorscale="RdBu", cmid=benchmarks["SPY buy-and-hold"]),
                  title=f"Sharpe by tenor pair from {info['pair_grid_first_signal']}; "
                        f"color midpoint is SPY buy-and-hold ({benchmarks['SPY buy-and-hold']:.2f})")
fig.show()
display(benchmarks.to_frame("Sharpe, same window"))

,"Sharpe, same window"
strategy,
SPY buy-and-hold,0.815
60/40 SPY/IEF,0.806
VIX < 20 filter,0.580
Vol target 15%,0.839


### Walk-forward pair selection

Each January the walk-forward holds whichever binary pair had the best Sharpe on all earlier data. This is the honest version of choosing a pair from the grid above: if it trails the fixed `3M/1M` rule or SPY over the same years, the in-sample best cell was luck.

In [ ]:
pair_returns = read("pair_returns.csv", index_col=0, parse_dates=True)
wf_name = "walk-forward binary"
wf_window = pair_returns[wf_name].dropna().index
compare = pair_returns.loc[wf_window, [wf_name, MAIN, "SPY buy-and-hold", "60/40 SPY/IEF"]]

wf_equity = (1.0 + compare).cumprod()
fig = go.Figure([go.Scatter(x=wf_equity.index, y=wf_equity[c], name=c) for c in compare])
fig.update_yaxes(type="log")
fig.update_layout(height=420, title=f"Walk-forward vs fixed rule, {wf_window[0].date()} onward")
fig.show()

years = (wf_window[-1] - wf_window[0]).days / 365.25
display(pd.DataFrame({
    "cagr": wf_equity.iloc[-1] ** (1 / years) - 1.0,
    "ann_vol": compare.std() * np.sqrt(252),
    "max_drawdown": (wf_equity / wf_equity.cummax() - 1.0).min(),
}))
display(read("walk_forward_selections.csv"))

,cagr,ann_vol,max_drawdown
walk-forward binary,0.100,0.141,-0.277
binary 3M/1M,0.122,0.139,-0.277
SPY buy-and-hold,0.138,0.175,-0.337
60/40 SPY/IEF,0.088,0.103,-0.212


,year,selected,trailing_sharpe
0,2015,binary 6M/1M,1.514
1,2016,binary 12M/6M,1.091
2,2017,binary 12M/6M,1.053
3,2018,binary 12M/6M,1.205
4,2019,binary 12M/6M,0.916
5,2020,binary 12M/9D,1.000
6,2021,binary 12M/9D,0.984
7,2022,binary 6M/1M,1.053
8,2023,binary 6M/1M,0.721
9,2024,binary 6M/1M,0.781


## Predictive regressions

Forward 21- and 63-day SPY excess log returns, and forward realized volatility, regressed on today's signal. Regressors are standardized, so each point is the effect of a one-standard-deviation move. Error bars are 95% intervals with Newey-West errors, because overlapping daily windows make plain t-statistics far too large.

The key comparison is `signal` against `signal + VIX`. If the coefficient collapses once the VIX level is added, the term structure carries little information beyond the level of fear.

In [ ]:
regressions = read("regressions.csv")
main_pair = info["main_pair"]
subset = regressions[regressions["pair"].isin([main_pair, "VIX level"])].copy()
subset["label"] = np.where(subset["pair"] == "VIX level", "VIX only", subset["signal"] + ": " + subset["spec"])

panels = [(t, h) for t in ["excess_return", "realized_vol"] for h in sorted(subset["horizon"].unique())]
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"{t.replace('_', ' ')}, {h} days" for t, h in panels],
                    vertical_spacing=0.2, horizontal_spacing=0.3)
for k, (target, horizon) in enumerate(panels):
    rows = subset[(subset["target"] == target) & (subset["horizon"] == horizon)]
    fig.add_trace(go.Scatter(x=rows["coef"], y=rows["label"], mode="markers", showlegend=False,
                             error_x=dict(type="data", symmetric=False,
                                          array=rows["ci_high"] - rows["coef"],
                                          arrayminus=rows["coef"] - rows["ci_low"])),
                  row=k // 2 + 1, col=k % 2 + 1)
    fig.add_vline(x=0, line_dash="dot", row=k // 2 + 1, col=k % 2 + 1)
fig.update_layout(height=700, title=f"{main_pair} coefficients per one standard deviation, 95% Newey-West intervals")
fig.show()

display(subset[["target", "horizon", "label", "coef", "t", "p", "r2", "nobs"]].sort_values(["target", "horizon"]))

,target,horizon,label,coef,t,p,r2,nobs
0,excess_return,21,VIX only,0.009,4.261,0.000,0.048,4195
25,excess_return,21,ratio: signal,-0.005,-2.040,0.041,0.015,4195
26,excess_return,21,ratio: signal + VIX,0.002,0.469,0.639,0.049,4195
27,excess_return,21,ratio: signal + VIX + RV21,0.002,0.467,0.641,0.049,4195
28,excess_return,21,z: signal,-0.003,-1.237,0.216,0.005,4017
29,excess_return,21,z: signal + VIX,0.003,0.951,0.342,0.056,4017
30,excess_return,21,z: signal + VIX + RV21,0.003,0.941,0.347,0.056,4017
61,excess_return,63,VIX only,0.021,4.307,0.000,0.105,4153
86,excess_return,63,ratio: signal,-0.015,-4.415,0.000,0.057,4153
87,excess_return,63,ratio: signal + VIX,-0.003,-0.513,0.608,0.106,4153


In [ ]:
oos = read("oos_r2.csv")
oos["label"] = np.where(oos["pair"] == "VIX level", "VIX level", oos["signal"] + " " + oos["pair"])
fig = make_subplots(rows=2, cols=1, subplot_titles=["Forward excess return", "Forward realized volatility"],
                    vertical_spacing=0.25)
for row, target in enumerate(["excess_return", "realized_vol"], start=1):
    for horizon in sorted(oos["horizon"].unique()):
        rows = oos[(oos["target"] == target) & (oos["horizon"] == horizon)].sort_values("label")
        fig.add_trace(go.Bar(x=rows["label"], y=rows["oos_r2"], name=f"{horizon} days",
                             showlegend=row == 1, legendgroup=str(horizon)), row=row, col=1)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=750, barmode="group",
                  title="Expanding-window out-of-sample R-squared vs the historical mean; above 0 beats the mean")
fig.show()

## Timing in stress and subperiod stability

`inversion_days_before_peak` counts trading days from the first daily inversion to the SPY peak; negative means the curve inverted after the selloff began. A negative `reentry_days_after_trough` means the monthly rule was fully invested again before the bottom, and `spy_loss_ridden_reentry_to_trough` is the decline it then absorbed. `spy_gain_missed_trough_to_reentry` is the rebound it sat out.

In [ ]:
display(read("lead_lag.csv"))

subperiods = read("subperiods.csv")
display(subperiods.pivot(index="strategy", columns="period", values="sharpe").loc[metrics.index])
display(subperiods.pivot(index="strategy", columns="period", values="max_drawdown").loc[metrics.index])

,window,peak,trough,spy_drawdown,first_inversion,inversion_days_before_peak,spy_move_peak_to_inversion,strategy_exit,spy_move_peak_to_exit,reentry_signal,strategy_reentry,reentry_days_after_trough,spy_gain_missed_trough_to_reentry,spy_loss_ridden_reentry_to_trough
0,Aug 2011 debt ceiling,2011-07-07,2011-10-03,-0.184,2011-06-16,14,-0.064,2011-08-01,-0.049,2011-10-05,2011-09-01,-21.000,NaN,-0.086
1,Aug 2015 China devaluation,2015-08-17,2015-08-25,-0.111,2015-06-29,34,-0.025,NaN,NaN,2015-09-02,NaN,NaN,NaN,NaN
2,Feb 2018 Volmageddon,2018-01-26,2018-02-08,-0.101,2018-02-02,-5,-0.039,NaN,NaN,2018-02-14,NaN,NaN,NaN,NaN
3,Q4 2018 selloff,2018-09-20,2018-12-24,-0.193,2018-10-10,-14,-0.048,2018-11-01,-0.064,2019-01-04,2018-12-03,-14.000,NaN,-0.156
4,Mar 2020 COVID crash,2020-02-19,2020-03-23,-0.337,2020-01-27,16,-0.044,2020-02-03,-0.042,2020-04-24,2020-05-01,28.000,0.268,NaN
5,2022 bear market,2022-01-03,2022-10-12,-0.245,2021-12-01,22,-0.060,2022-03-01,-0.100,2022-10-14,2022-04-01,-133.000,NaN,-0.206
6,Aug 2024 yen carry unwind,2024-07-16,2024-08-05,-0.084,2024-08-02,-13,-0.057,NaN,NaN,2024-08-08,NaN,NaN,NaN,NaN
7,Apr 2025 tariff shock,2025-02-19,2025-04-08,-0.188,2024-12-18,40,-0.047,2024-11-01,-0.071,2025-04-25,2024-12-02,-86.000,NaN,-0.172


period,2010-2014,2015-2019,2020-2026
strategy,,,
binary 3M/1M,1.019,0.677,0.798
zscore 3M/1M,0.812,0.530,0.496
bands 3M/1M,1.276,0.657,0.670
SPY buy-and-hold,1.116,0.804,0.679
60/40 SPY/IEF,1.444,0.964,0.556
VIX < 20 filter,1.057,0.352,0.413
Vol target 15%,1.190,0.743,0.690


period,2010-2014,2015-2019,2020-2026
strategy,,,
binary 3M/1M,-0.175,-0.209,-0.277
zscore 3M/1M,-0.097,-0.177,-0.252
bands 3M/1M,-0.097,-0.179,-0.259
SPY buy-and-hold,-0.186,-0.193,-0.337
60/40 SPY/IEF,-0.083,-0.106,-0.212
VIX < 20 filter,-0.117,-0.209,-0.152
Vol target 15%,-0.168,-0.169,-0.241
